# Pneumonia Detection from Chest X-Rays with a Custom CNN

**CSCK506 End-of-Module Assignment** — binary classification: **Normal vs Pneumonia**.

Upload this notebook to Google Colab. Use **Runtime → Change runtime type → GPU (T4)**.

| Section | Purpose |
|---|---|
| 0. Setup | Config, libraries, dataset location. Outputs save to Google Drive. |
| 1. EDA | Class balance, sample images, sizes, intensity |
| 2. Feature engineering | Resize, scale 0–1, optional CLAHE, mild aug, class weights |
| 3–4. Experiments | Baseline CNN + 6 one-factor alternatives, all on **80/20** |
| 5. Official test | Score the selected **80/20** custom CNN **once**, after validation review |
| 6. MobileNetV2 | Optional frozen ImageNet comparison on the same 80/20 split and official test |
| 7. 70/15/15 | Split comparison only: baseline CNN, then MobileNet if transfer learning is on |
| 8. Summary | Four-row table after every protocol has finished |

**Split protocol**
- **Baseline:** pool official train+val, split **80/20**; official 624-image test unused until the end.
- **Comparison:** pool all folders into **70/15/15**. Do not mix those test scores with official-test scores.

**Seven custom-CNN runs** (each alternative changes **one** setting from the baseline):

| Run | Change |
|---|---|
| 1 | Baseline: 150px, 4 blocks, ReLU, same padding, stride 1, mild aug (no flip) |
| 2 | Image size 224 |
| 3 | 3 conv–pool blocks (32/64/128) |
| 4 | LeakyReLU |
| 5 | Valid padding |
| 6 | Conv stride 2 (pooling padding stays on the baseline size-based rule) |
| 7 | Augmentation off |

Finish the **80/20** protocol first (runs 1–7, official test, optional MobileNet). All **70/15/15** runs are in section 7 at the end.

The **marked model** is still the custom CNN. MobileNetV2 is an extra comparison.


## 0. Setup and configuration

For the next baseline check keep all four flags **False**. After the baseline saves cleanly, turn on the controlled comparisons.

Checkpoints, result tables and training histories are written to Google Drive under `outputs/checkpoints`.


In [ ]:
# %pip install -q tensorflow matplotlib seaborn scikit-learn opencv-python-headless pandas

import os
import random
import shutil
import sys
import time
from copy import deepcopy
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IN_COLAB = "google.colab" in sys.modules
print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Colab:", IN_COLAB)


### Config

- Baseline split = **80/20** on official train+val; official test held out.
- `RUN_SPLIT_COMPARISON` trains 70/15/15 **at the end** (baseline CNN, and MobileNet if transfer learning is on). Not used to pick the official-test model.
- Mild augmentation is rotation / translation / zoom only (no horizontal flip).
- `EVALUATE_TEST_AFTER_ALL_RUNS = False` until validation results are reviewed.


In [ ]:
# -----------------------------------------------------------------------------
# CONFIG  (Larry notes: 80/20 hold-out test = baseline; 70/15/15 = comparison only)
# Runs 1-7 each change ONE setting from BASELINE (same 80/20 split). Official test once.
# -----------------------------------------------------------------------------
SEED = 42
COLOR_MODE = "grayscale"
BATCH_SIZE = 32
EPOCHS = 20
KERNEL_SIZE = 3
DROPOUT_RATE = 0.5
DENSE_UNITS = 128
USE_CLAHE = False
USE_CLASS_WEIGHTS = True
USE_HORIZONTAL_FLIP = False  # keep L/R anatomy; compare mild rot/shift/zoom only
SPLIT_BY_PATIENT = True
# Next baseline check: all False. Turn these on only after the baseline saves cleanly.
RUN_TRANSFER_LEARNING = False  # optional MobileNetV2 (80/20 first; 70/15/15 only in the end block)
RUN_ALL_EXPERIMENTS = False  # False = 80/20 baseline only
RUN_SPLIT_COMPARISON = False  # 70/15/15 CNN (+ MobileNet if TL on) in the last block only
EVALUATE_TEST_AFTER_ALL_RUNS = False  # review validation first; do not auto-score official test
TL_IMG_SIZE = 224
TL_EPOCHS = 10

# Baseline CNN (run 1)
BASELINE = {
    "img_size": 150,
    "n_blocks": 4,
    "filters": (32, 64, 128, 256),
    "activation": "relu",
    "padding": "same",
    "conv_strides": 1,
    "use_augmentation": True,
    "split_scheme": "80_20_holdout",  # pool official train+val, 80/20; keep official test
}

# Google Drive folder you uploaded (Colab path)
DATA_DIR = "/content/drive/MyDrive/Master of Liverpool/CSCK506_deep_learning/end_module/data"
KAGGLE_DATASET = "paultimothymooney/chest-xray-pneumonia"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    print("Drive mounted. DATA_DIR exists:", Path(DATA_DIR).exists())

# Checkpoints, result tables and histories go on Drive so a Colab disconnect does not wipe them.
if IN_COLAB:
    OUTPUT_DIR = Path(DATA_DIR).parent / "outputs"
else:
    OUTPUT_DIR = Path.cwd() / "outputs"
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

CHANNELS = 1 if COLOR_MODE == "grayscale" else 3
sns.set_theme(style="whitegrid")
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]


### Locate the dataset

Colab mounts Drive, then copies images to `/content/chest_xray_local` once so the training runs are not limited by Drive I/O.

In [ ]:
# -----------------------------------------------------------------------------
# Locate or download dataset
# -----------------------------------------------------------------------------
def find_data_root(start: Path):
    candidates = [
        start,
        start / "chest_xray",
        start / "chest_xray" / "chest_xray",
    ]
    for p in candidates:
        if (p / "train" / "NORMAL").exists() and (p / "train" / "PNEUMONIA").exists():
            return p
    if start.exists():
        for train_dir in start.rglob("train"):
            if (train_dir / "NORMAL").exists() and (train_dir / "PNEUMONIA").exists():
                return train_dir.parent
    return None


def download_from_kaggle(dest: Path) -> Path:
    dest.mkdir(parents=True, exist_ok=True)
    os.environ["KAGGLE_CONFIG_DIR"] = str(dest)
    if IN_COLAB:
        from google.colab import files

        print("Upload kaggle.json")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise FileNotFoundError("kaggle.json was not uploaded.")
        (dest / "kaggle.json").write_bytes(uploaded["kaggle.json"])
    else:
        src = Path.home() / ".kaggle" / "kaggle.json"
        if not src.exists():
            raise FileNotFoundError(
                "Put kaggle.json in ~/.kaggle/ or set DATA_DIR to an existing dataset folder."
            )
        (dest / "kaggle.json").write_bytes(src.read_bytes())
    try:
        os.chmod(dest / "kaggle.json", 0o600)
    except OSError:
        pass
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
    subprocess.check_call(
        [
            "kaggle",
            "datasets",
            "download",
            "-d",
            KAGGLE_DATASET,
            "-p",
            str(dest),
            "--unzip",
        ]
    )
    root = find_data_root(dest)
    if root is None:
        raise FileNotFoundError(f"Downloaded files but could not find train/NORMAL under {dest}")
    return root


search_roots = []
if DATA_DIR:
    search_roots.append(Path(DATA_DIR))
if IN_COLAB:
    search_roots.extend(
        [
            Path("/content/drive/MyDrive/Master of Liverpool/CSCK506_deep_learning/end_module/data"),
            Path("/content/drive/MyDrive/Master of Liverpool/CSCK506_deep_learning/end_module/data/chest_xray"),
            Path("/content/chest_xray"),
            Path("/content/data"),
        ]
    )
search_roots.extend(
    [
        Path.cwd() / "chest_xray",
        Path.cwd() / "data" / "chest_xray",
        Path.cwd() / "data",
    ]
)

DATA_ROOT = None
for p in search_roots:
    DATA_ROOT = find_data_root(p)
    if DATA_ROOT:
        break

if DATA_ROOT is None:
    print("Could not find train/NORMAL under DATA_DIR.")
    print("Looked at:", DATA_DIR)
    p = Path(DATA_DIR)
    if p.exists():
        print("Folder contents:", [x.name for x in p.iterdir()])
    else:
        print("That path does not exist. Check Drive folder names (spaces/spelling).")
    dest = Path("/content/data") if IN_COLAB else Path.cwd() / "data"
    print("Falling back to Kaggle download into", dest)
    DATA_ROOT = download_from_kaggle(dest)

# Drive I/O is slow for 7 training runs. Copy once onto Colab local disk.
if IN_COLAB:
    local_root = Path("/content/chest_xray_local")
    if not (local_root / "train" / "NORMAL").exists():
        print("Copying dataset to", local_root, "for faster training I/O...")
        shutil.copytree(DATA_ROOT, local_root, dirs_exist_ok=True)
    copied = find_data_root(local_root)
    if copied is not None:
        DATA_ROOT = copied

TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"
TEST_DIR = DATA_ROOT / "test"
print("DATA_ROOT:", DATA_ROOT.resolve())
for split in ("train", "val", "test"):
    print(split, "->", DATA_ROOT / split, "exists:", (DATA_ROOT / split).exists())


## 1. Exploratory data analysis

Official validation is tiny (~16 images), which is why the baseline does **not** use that folder as the tuning set.

In [ ]:
# -----------------------------------------------------------------------------
# 1. EDA
# -----------------------------------------------------------------------------
def list_images(folder: Path):
    exts = {".jpeg", ".jpg", ".png", ".JPEG", ".JPG", ".PNG"}
    return sorted(p for p in folder.rglob("*") if p.suffix in exts)


def split_counts(split_dir: Path):
    return {c: len(list_images(split_dir / c)) for c in CLASS_NAMES}


rows = []
for split, folder in (("train", TRAIN_DIR), ("val", VAL_DIR), ("test", TEST_DIR)):
    counts = split_counts(folder)
    for label, n in counts.items():
        rows.append({"split": split, "class": label, "count": n})

count_df = pd.DataFrame(rows)
pivot = count_df.pivot(index="split", columns="class", values="count").reindex(
    ["train", "val", "test"]
)
pivot["total"] = pivot.sum(axis=1)
print(pivot)
print("\nPneumonia share of each split:")
print((pivot["PNEUMONIA"] / pivot["total"]).map("{:.1%}".format))

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=count_df, x="split", y="count", hue="class", order=["train", "val", "test"], ax=ax)
ax.set_title("Class counts by split")
ax.set_xlabel("Split")
ax.set_ylabel("Number of images")
plt.tight_layout()
plt.show()

val_total = int(pivot.loc["val", "total"])
if val_total < 50:
    print(
        f"Note: official val has only {val_total} images. "
        "Baseline therefore pools official train+val and splits 80/20, "
        "keeping the official test set for a later one-shot evaluation. "
        "70/15/15 (pooling all three folders) is a comparison only."
    )


def show_samples(n_per_class: int = 5):
    fig, axes = plt.subplots(2, n_per_class, figsize=(2.4 * n_per_class, 5.2))
    for r, label in enumerate(CLASS_NAMES):
        files = list_images(TRAIN_DIR / label)
        sample = random.sample(files, k=min(n_per_class, len(files)))
        for c, path in enumerate(sample):
            img = Image.open(path)
            axes[r, c].imshow(img, cmap="gray")
            axes[r, c].axis("off")
            if c == 0:
                axes[r, c].set_ylabel(label, fontsize=11)
            axes[r, c].set_title(path.name[:22], fontsize=8)
    fig.suptitle("Random training samples (original files, not resized)", y=1.02)
    plt.tight_layout()
    plt.show()


show_samples()

size_rows = []
for label in CLASS_NAMES:
    files = list_images(TRAIN_DIR / label)
    sample = files if len(files) <= 400 else random.sample(files, 400)
    for path in sample:
        with Image.open(path) as im:
            w, h = im.size
        size_rows.append({"class": label, "width": w, "height": h, "aspect": w / h})

size_df = pd.DataFrame(size_rows)
print(size_df.groupby("class")[["width", "height", "aspect"]].describe().round(1))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
sns.histplot(data=size_df, x="width", hue="class", ax=axes[0], element="step")
sns.histplot(data=size_df, x="height", hue="class", ax=axes[1], element="step")
sns.histplot(data=size_df, x="aspect", hue="class", ax=axes[2], element="step")
axes[0].set_title("Width (px)")
axes[1].set_title("Height (px)")
axes[2].set_title("Aspect ratio W/H")
fig.suptitle("Original image geometry (sample of training files)")
plt.tight_layout()
plt.show()


def intensity_stats(n_per_class: int = 250) -> pd.DataFrame:
    recs = []
    for label in CLASS_NAMES:
        files = list_images(TRAIN_DIR / label)
        sample = files if len(files) <= n_per_class else random.sample(files, n_per_class)
        for path in sample:
            arr = np.asarray(Image.open(path).convert("L"), dtype=np.float32)
            recs.append({"class": label, "mean": float(arr.mean()), "std": float(arr.std())})
    return pd.DataFrame(recs)


intensity_df = intensity_stats()
print(intensity_df.groupby("class")[["mean", "std"]].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
sns.kdeplot(data=intensity_df, x="mean", hue="class", fill=True, ax=axes[0])
sns.kdeplot(data=intensity_df, x="std", hue="class", fill=True, ax=axes[1])
axes[0].set_title("Mean pixel intensity")
axes[1].set_title("Pixel std. deviation")
fig.suptitle("Grayscale intensity (training sample)")
plt.tight_layout()
plt.show()


def mean_image(label: str, n: int = 200) -> np.ndarray:
    files = list_images(TRAIN_DIR / label)
    sample = files if len(files) <= n else random.sample(files, n)
    acc = np.zeros((150, 150), dtype=np.float64)
    for path in sample:
        im = Image.open(path).convert("L").resize((150, 150))
        acc += np.asarray(im, dtype=np.float64)
    return (acc / len(sample)).astype(np.float32)


mean_normal = mean_image("NORMAL")
mean_pneu = mean_image("PNEUMONIA")
diff = mean_pneu - mean_normal

fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for ax, img, title in zip(
    axes,
    [mean_normal, mean_pneu, diff],
    ["Mean NORMAL", "Mean PNEUMONIA", "Pneumonia - Normal"],
):
    im = ax.imshow(img, cmap="gray" if title != "Pneumonia - Normal" else "bwr")
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle("Average resized (150x150) training images")
plt.tight_layout()
plt.show()

print(
    """
EDA takeaways:
- Imbalance: more pneumonia than normal in train. Report accuracy (brief) plus recall/F1.
- Official val is tiny (~16 images). Baseline: 80/20 on train+val, official test held out.
- 70/15/15 (all folders pooled) is reported as a split comparison, not the test protocol.
- Image sizes vary -> resize is required.
- Optional CLAHE can boost local contrast; off by default.
"""
)


## 2. Feature engineering and splits

- Resize to the run's image size, grayscale, divide by 255 (range 0–1) for the custom CNN.
- MobileNetV2 uses RGB 224×224 and its own `preprocess_input` (no /255).
- Random augmentation is applied **inside the model**, on training only.

In [ ]:
# -----------------------------------------------------------------------------
# 2. Feature engineering + splits
# Larry notes: baseline = 80/20 on official train+val; official test held out.
# 70/15/15 (all folders pooled) is a split comparison only — do not mix test scores.
# Runs 1–7 each change ONE setting from BASELINE (not sequential winner-keep).
# -----------------------------------------------------------------------------
ckpt_dir = OUTPUT_DIR / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("Checkpoints / CSVs / histories:", ckpt_dir.resolve())

_printed_official_overlap = False


def patient_key(path: Path) -> str:
    """Stable grouping key from Kaggle filenames.

    Pneumonia files look like person30_bacteria_1.jpeg -> person30.
    Normal files look like IM-0115-0001.jpeg or NORMAL2-IM-0129-0001.jpeg;
    we drop only the last '-NNNN' view suffix.
    """
    stem = path.stem.lower()
    if stem.startswith("person"):
        return stem.split("_")[0]
    if "-" in stem:
        return stem.rsplit("-", 1)[0]
    return stem


def diagnose_patient_key_overlap(pool_df: pd.DataFrame, test_df: pd.DataFrame):
    """Confirm whether official-test overlap is a real Kaggle leak or a parse artefact."""
    pool_keys = set(pool_df["group"])
    test_keys = set(test_df["group"])
    leaked = sorted(pool_keys & test_keys)
    print("\n========== PATIENT_KEY() OVERLAP CHECK ==========")
    print(f"Unique keys: train/val pool={len(pool_keys)}, official test={len(test_keys)}")
    print(f"Keys in BOTH official train/val and official test: {len(leaked)}")
    print("(Original Kaggle split — not caused by our 80/20 patient split.)")

    def bucket(k: str) -> str:
        k = str(k)
        if k.startswith("person"):
            return "person_* (pneumonia-style patient ID)"
        if k.startswith("im-") or "-im-" in k:
            return "IM-* (normal-style filename stem)"
        return "other"

    if not leaked:
        print("No overlapping keys.")
        return leaked

    print("\nOverlap by filename pattern:")
    print(pd.Series([bucket(k) for k in leaked]).value_counts().to_string())

    both = pd.concat(
        [
            pool_df.assign(source="train_val_pool"),
            test_df.assign(source="official_test"),
        ],
        ignore_index=True,
    )
    person_keys = [k for k in leaked if str(k).startswith("person")]
    other_keys = [k for k in leaked if not str(k).startswith("person")]

    def show_examples(keys, n=4):
        for k in keys[:n]:
            print(f"\n  key={k}")
            rows = both[both["group"] == k]
            for _, r in rows.head(8).iterrows():
                print(f"    [{r['source']}] {CLASS_NAMES[int(r['label'])]}  {Path(r['path']).name}")

    n_same_name = 0
    n_diff_files = 0
    for k in leaked:
        pool_names = {Path(p).name.lower() for p in pool_df.loc[pool_df["group"] == k, "path"]}
        test_names = {Path(p).name.lower() for p in test_df.loc[test_df["group"] == k, "path"]}
        if pool_names & test_names:
            n_same_name += 1
        if pool_names - test_names or test_names - pool_names:
            n_diff_files += 1

    print(f"\nKeys whose exact filename appears on both sides: {n_same_name}")
    print(f"Keys with different filenames on each side (typical multi-image patient): {n_diff_files}")

    if person_keys:
        print(
            f"\nExamples of overlapping person_* keys ({len(person_keys)}). "
            "Same person ID in Kaggle train and test = real patient leak:"
        )
        show_examples(person_keys)
    if other_keys:
        print(
            f"\nExamples of overlapping non-person keys ({len(other_keys)}). "
            "If these look like unrelated IM-* stems, patient_key() may be too coarse:"
        )
        show_examples(other_keys)

    if person_keys and len(person_keys) >= max(1, 0.5 * len(leaked)):
        print(
            "\nConclusion: most overlap is person_* IDs, so this is the original Kaggle "
            "patient leak. Our 80/20 split only groups patients inside official train+val; "
            "the official test is left unchanged for literature comparison."
        )
    else:
        print(
            "\nConclusion: many overlapping keys are not person_* IDs. Inspect the examples "
            "above before treating them as true patient leak."
        )
    return leaked


def collect_labeled_files(split_dirs):
    rows = []
    for split_dir in split_dirs:
        for label_i, label in enumerate(CLASS_NAMES):
            for p in list_images(split_dir / label):
                rows.append({"path": p, "label": label_i, "group": patient_key(p)})
    return pd.DataFrame(rows)


def _group_or_row_split(df: pd.DataFrame, test_size: float):
    if SPLIT_BY_PATIENT:
        patient_lab = df.groupby("group")["label"].agg(lambda s: int(s.mode().iloc[0]))
        left_g, right_g = train_test_split(
            patient_lab.index,
            test_size=test_size,
            stratify=patient_lab.values,
            random_state=SEED,
        )
        return df[df["group"].isin(left_g)].copy(), df[df["group"].isin(right_g)].copy()
    left, right = train_test_split(
        df, test_size=test_size, stratify=df["label"], random_state=SEED
    )
    return left.copy(), right.copy()


def make_80_20_holdout():
    """Pool official train+val, 80/20; keep official test (624) untouched."""
    global _printed_official_overlap
    pool = collect_labeled_files([TRAIN_DIR, VAL_DIR])
    test_df = collect_labeled_files([TEST_DIR])
    train_df, val_df = _group_or_row_split(pool, test_size=0.20)
    if not _printed_official_overlap:
        diagnose_patient_key_overlap(pool, test_df)
        _printed_official_overlap = True
    return train_df, val_df, test_df, "80/20 on train+val; official test held out"


def make_70_15_15():
    """Comparison only: pool all three folders. Test images are NOT the official Kaggle test."""
    all_df = collect_labeled_files([TRAIN_DIR, VAL_DIR, TEST_DIR])
    train_df, rest_df = _group_or_row_split(all_df, test_size=0.30)
    val_df, test_df = _group_or_row_split(rest_df, test_size=0.50)
    return train_df, val_df, test_df, "70/15/15 pooled (comparison; test is NOT official Kaggle test)"


def print_split_table(name, train_df, val_df, test_df):
    rows = []
    for split, part in (("train", train_df), ("val", val_df), ("test", test_df)):
        n0 = int((part["label"] == 0).sum())
        n1 = int((part["label"] == 1).sum())
        rows.append({"split": split, "NORMAL": n0, "PNEUMONIA": n1, "total": n0 + n1})
    tab = pd.DataFrame(rows).set_index("split")
    print(f"\n{name}")
    print(tab)
    print("Pneumonia share:")
    print((tab["PNEUMONIA"] / tab["total"]).map("{:.1%}".format))
    return tab


def class_weights_from_train(train_df):
    y = train_df["label"].to_numpy(dtype=np.int32)
    values = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=y)
    weights = {0: float(values[0]), 1: float(values[1])}
    print("Class weights from this run's training labels only:", weights)
    if not USE_CLASS_WEIGHTS:
        return None
    return weights


def clahe_numpy(gray_float01: np.ndarray) -> np.ndarray:
    if gray_float01.ndim == 3:
        gray_float01 = gray_float01[..., 0]
    u8 = np.clip(gray_float01 * 255.0, 0, 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    out = clahe.apply(u8).astype(np.float32) / 255.0
    return out[..., None]


demo_size = BASELINE["img_size"]
fig, axes = plt.subplots(2, 2, figsize=(7, 7))
for i, label in enumerate(CLASS_NAMES):
    path = random.choice(list_images(TRAIN_DIR / label))
    raw = (
        np.asarray(Image.open(path).convert("L").resize((demo_size, demo_size)), dtype=np.float32)
        / 255.0
    )
    enhanced = clahe_numpy(raw)[..., 0]
    axes[i, 0].imshow(raw, cmap="gray")
    axes[i, 0].set_title(f"{label} - scaled")
    axes[i, 1].imshow(enhanced, cmap="gray")
    axes[i, 1].set_title(f"{label} - CLAHE")
    axes[i, 0].axis("off")
    axes[i, 1].axis("off")
fig.suptitle("Optional contrast enhancement (not used unless USE_CLAHE=True)")
plt.tight_layout()
plt.show()


def make_augmentation():
    aug_layers = [
        layers.RandomRotation(0.05),
        layers.RandomTranslation(0.05, 0.05),
        layers.RandomZoom(0.10),
    ]
    if USE_HORIZONTAL_FLIP:
        aug_layers.insert(0, layers.RandomFlip("horizontal"))
    return keras.Sequential(aug_layers, name="mild_augmentation")


def apply_activation(x, name, activation: str):
    if activation == "leaky_relu":
        return layers.LeakyReLU(0.1, name=name)(x)
    return layers.Activation(activation, name=name)(x)


def make_dataset_from_paths(paths, labels, shuffle: bool, batch_size: int, img_size: int):
    path_ds = tf.data.Dataset.from_tensor_slices(
        ([str(p) for p in paths], np.asarray(labels, dtype=np.float32))
    )
    channels = CHANNELS
    rescale = layers.Rescaling(1.0 / 255.0)
    use_clahe = USE_CLAHE

    def _clahe_np(img):
        img = img.astype(np.float32)
        if channels == 3:
            gray = cv2.cvtColor((np.clip(img, 0, 1) * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
            out = clahe_numpy(gray.astype(np.float32) / 255.0)
            return np.repeat(out, 3, axis=-1).astype(np.float32)
        return clahe_numpy(img).astype(np.float32)

    def _load(path, label):
        img = tf.io.read_file(path)
        img = tf.io.decode_image(img, channels=channels, expand_animations=False)
        img.set_shape([None, None, channels])
        img = tf.image.resize(img, [img_size, img_size])
        img = tf.cast(img, tf.float32)
        img = rescale(img)
        if use_clahe:
            img = tf.numpy_function(_clahe_np, [img], tf.float32)
            img.set_shape([img_size, img_size, channels])
        return img, label

    ds = path_ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=1024, seed=SEED)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def make_rgb_dataset_from_paths(paths, labels, shuffle: bool, batch_size: int, img_size: int):
    """RGB 0–255 tensors for MobileNetV2 preprocess_input. No /255 rescale."""
    path_ds = tf.data.Dataset.from_tensor_slices(
        ([str(p) for p in paths], np.asarray(labels, dtype=np.float32))
    )

    def _load(path, label):
        img = tf.io.read_file(path)
        img = tf.io.decode_image(img, channels=1, expand_animations=False)
        img.set_shape([None, None, 1])
        img = tf.image.resize(img, [img_size, img_size])
        img = tf.image.grayscale_to_rgb(img)
        img = tf.cast(img, tf.float32)
        return img, label

    ds = path_ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=1024, seed=SEED)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def build_custom_cnn(cfg: dict) -> keras.Model:
    img_size = cfg["img_size"]
    inputs = keras.Input(shape=(img_size, img_size, CHANNELS), name="xray")
    x = inputs
    if cfg["use_augmentation"]:
        x = make_augmentation()(x)
    n_blocks = min(cfg["n_blocks"], len(cfg["filters"]))
    for i in range(n_blocks):
        name = f"block{i + 1}"
        x = layers.Conv2D(
            cfg["filters"][i],
            KERNEL_SIZE,
            strides=cfg["conv_strides"],
            padding=cfg["padding"],
            use_bias=False,
            name=f"{name}_conv",
        )(x)
        x = layers.BatchNormalization(name=f"{name}_bn")(x)
        x = apply_activation(x, f"{name}_act", cfg["activation"])
        h = x.shape[1]
        if h is None:
            x = layers.MaxPooling2D(2, strides=2, padding="same", name=f"{name}_pool")(x)
        elif int(h) >= 2:
            # Pooling padding follows spatial size only, not conv stride, so Run 6
            # remains a one-factor stride comparison.
            pool_padding = "same" if int(h) < 4 else "valid"
            x = layers.MaxPooling2D(2, strides=2, padding=pool_padding, name=f"{name}_pool")(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(DROPOUT_RATE, name="dropout_1")(x)
    x = layers.Dense(DENSE_UNITS, name="dense")(x)
    x = apply_activation(x, "dense_act", cfg["activation"])
    x = layers.Dropout(DROPOUT_RATE, name="dropout_2")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="pneumonia_prob")(x)
    return keras.Model(inputs, outputs, name="custom_cnn")


def collect_labels(model, ds, threshold: float = 0.5):
    y_true, y_prob = [], []
    for xb, yb in ds:
        y_true.append(yb.numpy().reshape(-1))
        y_prob.append(model.predict(xb, verbose=0).reshape(-1))
    y_true = np.concatenate(y_true)
    y_prob = np.concatenate(y_prob)
    y_pred = (y_prob >= threshold).astype(int)
    return y_true, y_prob, y_pred


def select_threshold(y_true, y_prob):
    """Youden's J on validation: maximise recall + specificity - 1. Never fit this on test."""
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    if len(np.unique(y_true)) < 2:
        return 0.5, {"youden_j": float("nan")}
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    finite = np.isfinite(thresholds)
    fpr, tpr, thresholds = fpr[finite], tpr[finite], thresholds[finite]
    if len(thresholds) == 0:
        return 0.5, {"youden_j": float("nan")}
    j = tpr - fpr
    i = int(np.argmax(j))
    thr = float(np.clip(thresholds[i], 0.0, 1.0))
    return thr, {
        "youden_j": float(j[i]),
        "tpr_at_thr": float(tpr[i]),
        "tnr_at_thr": float(1.0 - fpr[i]),
    }


def print_clf_report(y_true, y_pred):
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4, zero_division=0))


def split_metrics(y_true, y_prob, y_pred, prefix: str):
    prob = np.clip(y_prob, 1e-7, 1.0 - 1e-7)
    return {
        f"{prefix}_loss": float(log_loss(y_true, prob)),
        f"{prefix}_accuracy": float(accuracy_score(y_true, y_pred)),
        f"{prefix}_precision": float(precision_score(y_true, y_pred, zero_division=0)),
        f"{prefix}_recall": float(recall_score(y_true, y_pred, zero_division=0)),
        f"{prefix}_specificity": float(recall_score(y_true, y_pred, pos_label=0, zero_division=0)),
        f"{prefix}_f1": float(f1_score(y_true, y_pred, zero_division=0)),
        f"{prefix}_auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else float("nan"),
    }


def persist_run_outputs(row: dict, hist=None, y_true=None, y_prob=None):
    """Write this run's files immediately so a later crash does not lose completed work."""
    run_id = row["run"]
    if hist is not None and len(hist):
        hist_path = ckpt_dir / f"run{run_id}_history.csv"
        hist.to_csv(hist_path, index=False)
        print("Saved", hist_path)
    if y_true is not None and y_prob is not None:
        pred_path = ckpt_dir / f"run{run_id}_val_predictions.csv"
        pd.DataFrame({"y_true": y_true, "y_prob": y_prob}).to_csv(pred_path, index=False)
        print("Saved", pred_path)
    csv_path = ckpt_dir / "experiment_val_results.csv"
    new_df = pd.DataFrame([row])
    if csv_path.exists():
        old = pd.read_csv(csv_path)
        old = old[old["run"].astype(str) != str(run_id)]
        out = pd.concat([old, new_df], ignore_index=True)
    else:
        out = new_df
    out.to_csv(csv_path, index=False)
    print("Saved", csv_path)


def merge_cfg(overrides: dict) -> dict:
    cfg = deepcopy(BASELINE)
    cfg.update(overrides)
    return cfg


def train_one_run(run_id: str, title: str, overrides: dict):
    cfg = merge_cfg(overrides)
    print("\n" + "=" * 72)
    print(f"RUN {run_id}: {title}")
    print(cfg)
    print("=" * 72)

    if cfg["split_scheme"] == "80_20_holdout":
        train_df, val_df, test_df, split_name = make_80_20_holdout()
    else:
        train_df, val_df, test_df, split_name = make_70_15_15()
    print_split_table(split_name, train_df, val_df, test_df)
    class_weight = class_weights_from_train(train_df)

    img_size = cfg["img_size"]
    train_ds = make_dataset_from_paths(
        train_df["path"].tolist(), train_df["label"].tolist(), True, BATCH_SIZE, img_size
    )
    val_ds = make_dataset_from_paths(
        val_df["path"].tolist(), val_df["label"].tolist(), False, BATCH_SIZE, img_size
    )
    test_ds = make_dataset_from_paths(
        test_df["path"].tolist(), test_df["label"].tolist(), False, BATCH_SIZE, img_size
    )

    keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    try:
        model = build_custom_cnn(cfg)
    except Exception as exc:
        print("MODEL BUILD FAILED:", exc)
        row = {"run": run_id, "title": title, "error": str(exc), **{k: str(v) for k, v in cfg.items()}}
        persist_run_outputs(row)
        return row

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc"),
        ],
    )
    if str(run_id) == "1":
        model.summary()

    ckpt_path = ckpt_dir / f"run{run_id}_best.keras"
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True, verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=str(ckpt_path), monitor="val_loss", save_best_only=True, verbose=0
        ),
    ]
    t0 = time.perf_counter()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=1,
    )
    training_seconds = float(time.perf_counter() - t0)
    hist = pd.DataFrame(history.history)
    best_epoch = int(hist["val_loss"].idxmin()) + 1 if len(hist) else None

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    axes[0].plot(hist["loss"], label="train")
    axes[0].plot(hist["val_loss"], label="val")
    axes[0].set_title(f"Run {run_id} loss")
    axes[0].legend()
    axes[1].plot(hist["accuracy"], label="train")
    axes[1].plot(hist["val_accuracy"], label="val")
    axes[1].set_title(f"Run {run_id} accuracy")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    y_true, y_prob, y_pred_05 = collect_labels(model, val_ds, threshold=0.5)
    metrics_05 = split_metrics(y_true, y_prob, y_pred_05, "val05")
    print("\nValidation at threshold 0.5:")
    print_clf_report(y_true, y_pred_05)
    print({k: round(v, 4) for k, v in metrics_05.items()})
    if metrics_05["val05_specificity"] == 0 and metrics_05["val05_recall"] == 1:
        print(
            "Note: threshold 0.5 predicted Pneumonia for every validation image "
            "(0% Normal specificity). A validation-selected threshold is required."
        )

    threshold, thr_info = select_threshold(y_true, y_prob)
    y_pred = (y_prob >= threshold).astype(int)
    metrics_thr = split_metrics(y_true, y_prob, y_pred, "val")
    print(f"\nValidation-selected threshold (Youden J) = {threshold:.4f}  {thr_info}")
    print_clf_report(y_true, y_pred)
    print({k: round(v, 4) for k, v in metrics_thr.items()})

    row = {
        "run": run_id,
        "title": title,
        "split": cfg["split_scheme"],
        "img_size": cfg["img_size"],
        "n_blocks": cfg["n_blocks"],
        "activation": cfg["activation"],
        "padding": cfg["padding"],
        "conv_strides": cfg["conv_strides"],
        "use_augmentation": cfg["use_augmentation"],
        "best_epoch": best_epoch,
        "training_seconds": round(training_seconds, 1),
        "threshold": threshold,
        "val_loss_keras": float(hist["val_loss"].min()) if len(hist) else float("nan"),
        "error": "",
        **metrics_thr,
        "val_accuracy_at_0.5": metrics_05["val05_accuracy"],
        "val_recall_at_0.5": metrics_05["val05_recall"],
        "val_specificity_at_0.5": metrics_05["val05_specificity"],
        "val_f1_at_0.5": metrics_05["val05_f1"],
    }
    print(
        "Val summary (tuned threshold):",
        {k: round(v, 4) if isinstance(v, float) else v for k, v in row.items() if k.startswith("val_") or k in ("threshold", "training_seconds")},
    )

    if cfg["split_scheme"] != "80_20_holdout":
        yt, yp, _ = collect_labels(model, test_ds, threshold=0.5)
        yhat = (yp >= threshold).astype(int)
        extra = split_metrics(yt, yp, yhat, "own_test")
        row.update(extra)
        print("\n70/15/15 own-test (NOT the official Kaggle test — do not mix with official test):")
        print({k: round(v, 4) for k, v in extra.items()})
        print_clf_report(yt, yhat)

    persist_run_outputs(row, hist=hist, y_true=y_true, y_prob=y_prob)
    return row


## 3–4. Train the seven 80/20 comparisons

Each of runs 2–7 changes **one** knob from the baseline. Run 6 changes conv stride only; pooling padding no longer depends on stride.

Selection uses **validation loss** on the 80/20 split. Metrics are also reported at a **validation-selected threshold** (Youden J) because a 0.5 cut-off can classify every image as Pneumonia.

With `RUN_ALL_EXPERIMENTS = False` only the 80/20 baseline is trained. **70/15/15 is not in this cell** — it runs in section 7 after official test and MobileNet 80/20.


In [ ]:
# -----------------------------------------------------------------------------
# 3-4. Seven one-factor comparisons on the 80/20 split only
# Each of runs 2-7 changes one knob from BASELINE.
# -----------------------------------------------------------------------------
EXPERIMENT_PLAN = [
    ("1", "Baseline", {}),
    ("2", "Alternative image size 224", {"img_size": 224}),
    ("3", "Alternative number of blocks (3)", {"n_blocks": 3, "filters": (32, 64, 128)}),
    ("4", "Alternative activation (LeakyReLU)", {"activation": "leaky_relu"}),
    ("5", "Alternative padding (valid)", {"padding": "valid"}),
    ("6", "Alternative stride (2)", {"conv_strides": 2}),
    ("7", "Augmentation off", {"use_augmentation": False}),
]

SPLIT_COMPARISON = ("70_15_15", "Split comparison 70/15/15", {"split_scheme": "70_15_15"})

print(
    """
Comparison plan (Larry notes; each alternative vs baseline, same 80/20 split):
  Run 1  Baseline: 150px, 4 blocks, ReLU, same padding, stride 1, mild aug (no flip)
  Run 2  Image size 224
  Run 3  3 conv-pool blocks (32/64/128)
  Run 4  LeakyReLU
  Run 5  Valid padding
  Run 6  Conv stride 2 (all conv layers; pooling padding unchanged from baseline rule)
  Run 7  Augmentation off
Then: official test once on the 80/20 run with the lowest validation loss (after review).
Then: optional MobileNetV2 on the same 80/20 split.
Then: all 70/15/15 comparisons in a separate end block.
"""
)

preview_train, _, _, _ = make_80_20_holdout()
preview_ds = make_dataset_from_paths(
    preview_train["path"].tolist(),
    preview_train["label"].tolist(),
    True,
    BATCH_SIZE,
    BASELINE["img_size"],
)
xb, yb = next(iter(preview_ds))
print(
    "Example training batch:",
    xb.shape,
    yb.shape,
    "pixel range",
    float(tf.reduce_min(xb)),
    float(tf.reduce_max(xb)),
)
fig, axes = plt.subplots(2, 6, figsize=(12, 4.2))
shown = {0: 0, 1: 0}
for images, labels in preview_ds.take(8):
    for img, lab in zip(images.numpy(), labels.numpy().reshape(-1)):
        k = int(lab)
        if shown[k] >= 6:
            continue
        ax = axes[k, shown[k]]
        ax.imshow(img.squeeze() if CHANNELS == 1 else img, cmap="gray" if CHANNELS == 1 else None)
        ax.axis("off")
        if shown[k] == 0:
            ax.set_ylabel(CLASS_NAMES[k])
        shown[k] += 1
    if shown[0] >= 6 and shown[1] >= 6:
        break
fig.suptitle(
    f"Training tensors after FE ({BASELINE['img_size']}x{BASELINE['img_size']}, {COLOR_MODE}, scaled)"
)
plt.tight_layout()
plt.show()

plan = list(EXPERIMENT_PLAN)
if not RUN_ALL_EXPERIMENTS:
    plan = [EXPERIMENT_PLAN[0]]
    print("RUN_ALL_EXPERIMENTS=False: training 80/20 baseline only.")
print("70/15/15 is not trained here. It runs in section 7 after 80/20 official test and MobileNet.")

results_rows = []
for run_id, title, overrides in plan:
    results_rows.append(train_one_run(run_id, title, overrides))

results_df = pd.DataFrame(results_rows)
print("\n========== VALIDATION COMPARISON TABLE ==========")
cols = [
    "run",
    "title",
    "split",
    "img_size",
    "n_blocks",
    "activation",
    "padding",
    "conv_strides",
    "use_augmentation",
    "training_seconds",
    "threshold",
    "val_loss_keras",
    "val_accuracy",
    "val_recall",
    "val_specificity",
    "val_f1",
    "val_auc",
    "val_specificity_at_0.5",
    "error",
]
show_cols = [c for c in cols if c in results_df.columns]
print(results_df[show_cols].to_string(index=False))
print("Saved", ckpt_dir / "experiment_val_results.csv")

holdout = results_df[(results_df["split"] == "80_20_holdout") & (results_df["error"].fillna("") == "")]
if len(holdout):
    winner = holdout.loc[holdout["val_loss_keras"].idxmin()]
    print(
        "\nSelected 80/20 model by lowest val_loss:",
        winner["run"],
        winner["title"],
        "val_loss=",
        winner["val_loss_keras"],
    )
    print("Also check val_recall (pneumonia) and val_specificity (Normal) before locking this choice.")
    if "threshold" in winner:
        print("Winner validation threshold:", winner["threshold"])
else:
    winner = None

cnn_test_row = None


## 5. Official test (once, custom CNN)

Loads the selected **80/20** checkpoint. Run this after reviewing 80/20 validation. 70/15/15 has not been trained yet.

In [ ]:
# -----------------------------------------------------------------------------
# 5. Official test — once, after all 80/20 validation decisions
# -----------------------------------------------------------------------------
if EVALUATE_TEST_AFTER_ALL_RUNS and winner is not None:
    print("\nLoading selected 80/20 checkpoint for official test (not retraining).")
    win_overrides = {}
    for run_id, title, overrides in EXPERIMENT_PLAN:
        if str(run_id) == str(winner["run"]):
            win_overrides = overrides
            break
    cfg = merge_cfg(win_overrides)
    _, _, test_df, _ = make_80_20_holdout()
    test_ds = make_dataset_from_paths(
        test_df["path"].tolist(), test_df["label"].tolist(), False, BATCH_SIZE, cfg["img_size"]
    )
    ckpt_file = ckpt_dir / f"run{winner['run']}_best.keras"
    try:
        model = keras.models.load_model(ckpt_file)
    except Exception as exc:
        print("load_model failed, rebuilding and loading weights:", exc)
        keras.backend.clear_session()
        model = build_custom_cnn(cfg)
        model.load_weights(ckpt_file)

    y_true, y_prob, _ = collect_labels(model, test_ds, threshold=0.5)
    win_thr = float(winner["threshold"]) if "threshold" in winner and pd.notna(winner["threshold"]) else 0.5
    y_pred = (y_prob >= win_thr).astype(int)
    cnn_test_row = split_metrics(y_true, y_prob, y_pred, "test")
    cnn_test_row["test_threshold"] = win_thr
    print("\n========== OFFICIAL TEST (once, after all 80/20 comparisons) ==========")
    print(f"Using validation-selected threshold {win_thr:.4f} (not retuned on test).")
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in cnn_test_row.items()})
    print_clf_report(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=axes[0],
    )
    axes[0].set_title("Confusion matrix (official test)")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("True")
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    axes[1].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_true, y_prob):.3f}")
    axes[1].plot([0, 1], [0, 1], "--", color="gray")
    axes[1].set_title("ROC (official test)")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    xt, yt = next(iter(test_ds))
    probs = model.predict(xt, verbose=0).reshape(-1)
    n_show = min(8, xt.shape[0])
    fig, axes = plt.subplots(1, n_show, figsize=(1.8 * n_show, 2.6))
    if n_show == 1:
        axes = [axes]
    for i in range(n_show):
        axes[i].imshow(xt[i].numpy().squeeze(), cmap="gray" if CHANNELS == 1 else None)
        true = CLASS_NAMES[int(yt[i].numpy().item())]
        pred = CLASS_NAMES[int(probs[i] >= win_thr)]
        axes[i].set_title(f"{true}\n{probs[i]:.2f} {pred}", fontsize=8)
        axes[i].axis("off")
    fig.suptitle("Official test batch: true class / P(pneumonia) / predicted", y=1.08)
    plt.tight_layout()
    plt.show()
elif not EVALUATE_TEST_AFTER_ALL_RUNS:
    print("EVALUATE_TEST_AFTER_ALL_RUNS=False: official test skipped.")

print(
    """
Notes for the report (Larry protocol):
- Baseline split = 80/20 on official train+val; official test unused until the end.
- 70/15/15 is a split comparison only (different test images; do not mix test scores).
- Baseline CNN: 150px, 4 blocks, ReLU, same padding, stride 1, mild aug without flip.
- Runs 2-7 change ONE setting from that baseline. Run 6 changes conv stride only;
  pooling padding no longer depends on stride.
- Pick the 80/20 winner with validation loss (also look at pneumonia recall and Normal specificity).
- Decision threshold is chosen on validation (Youden J), then frozen for official test.
- MobileNetV2 on 80/20 is an extra comparison, not the required custom CNN.
- 70/15/15 (CNN and optional MobileNet) runs after this, in section 7.
"""
)


## 6. MobileNetV2 on 80/20 (comparison only)

Frozen ImageNet backbone, 224 RGB, same mild augmentation (no flip). Optional. Trained on the **80/20** split and scored on the **official test**. The 70/15/15 MobileNet run is in section 7.


In [ ]:
# -----------------------------------------------------------------------------
# 6. MobileNetV2 (optional comparison, not the required custom CNN)
# Same 80/20 train/val as the custom CNN; official test scored once.
# 70/15/15 MobileNet is in section 7.
# -----------------------------------------------------------------------------
tl_val_row = None
tl_test_row = None

if not RUN_TRANSFER_LEARNING:
    print("Transfer learning skipped. Set RUN_TRANSFER_LEARNING = True to train MobileNetV2.")
else:
    print("\n" + "=" * 72)
    print("MOBILENETV2: frozen ImageNet backbone, 224 RGB, same 80/20 split")
    print("=" * 72)
    train_df, val_df, test_df, split_name = make_80_20_holdout()
    print_split_table(split_name + " (MobileNetV2)", train_df, val_df, test_df)
    class_weight = class_weights_from_train(train_df)

    tl_train = make_rgb_dataset_from_paths(
        train_df["path"].tolist(), train_df["label"].tolist(), True, BATCH_SIZE, TL_IMG_SIZE
    )
    tl_val = make_rgb_dataset_from_paths(
        val_df["path"].tolist(), val_df["label"].tolist(), False, BATCH_SIZE, TL_IMG_SIZE
    )
    tl_test = make_rgb_dataset_from_paths(
        test_df["path"].tolist(), test_df["label"].tolist(), False, BATCH_SIZE, TL_IMG_SIZE
    )

    keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    tl_base = keras.applications.MobileNetV2(
        include_top=False, weights="imagenet", input_shape=(TL_IMG_SIZE, TL_IMG_SIZE, 3)
    )
    tl_base.trainable = False
    inputs = keras.Input(shape=(TL_IMG_SIZE, TL_IMG_SIZE, 3), name="xray_rgb")
    x = make_augmentation()(inputs)
    x = keras.applications.mobilenet_v2.preprocess_input(x)
    x = tl_base(x, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="pneumonia_prob")(x)
    tl_model = keras.Model(inputs, outputs, name="mobilenetv2_frozen")
    tl_model.compile(
        optimizer=keras.optimizers.Adam(1e-4),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc"),
        ],
    )
    tl_model.summary()

    tl_ckpt = ckpt_dir / "mobilenetv2_80_20_best.keras"
    t0 = time.perf_counter()
    tl_history = tl_model.fit(
        tl_train,
        validation_data=tl_val,
        epochs=TL_EPOCHS,
        class_weight=class_weight,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=3, restore_best_weights=True, verbose=1
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1
            ),
            keras.callbacks.ModelCheckpoint(
                filepath=str(tl_ckpt), monitor="val_loss", save_best_only=True, verbose=0
            ),
        ],
        verbose=1,
    )
    tl_seconds = float(time.perf_counter() - t0)
    tl_hist = pd.DataFrame(tl_history.history)
    tl_hist.to_csv(ckpt_dir / "mobilenetv2_80_20_history.csv", index=False)
    print("Saved", ckpt_dir / "mobilenetv2_80_20_history.csv")
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    axes[0].plot(tl_hist["loss"], label="train")
    axes[0].plot(tl_hist["val_loss"], label="val")
    axes[0].set_title("MobileNetV2 loss")
    axes[0].legend()
    axes[1].plot(tl_hist["accuracy"], label="train")
    axes[1].plot(tl_hist["val_accuracy"], label="val")
    axes[1].set_title("MobileNetV2 accuracy")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    yv, pv, yhat_05 = collect_labels(tl_model, tl_val, threshold=0.5)
    tl_thr, tl_thr_info = select_threshold(yv, pv)
    yhat = (pv >= tl_thr).astype(int)
    tl_val_row = split_metrics(yv, pv, yhat, "val")
    tl_val_row["threshold"] = tl_thr
    tl_val_row["training_seconds"] = round(tl_seconds, 1)
    print("\nMobileNetV2 validation (80/20, not official test):")
    print(f"threshold={tl_thr:.4f}  {tl_thr_info}")
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in tl_val_row.items()})
    print_clf_report(yv, yhat)
    pd.DataFrame([tl_val_row]).to_csv(ckpt_dir / "mobilenetv2_80_20_val.csv", index=False)

    if EVALUATE_TEST_AFTER_ALL_RUNS:
        yt, pt, _ = collect_labels(tl_model, tl_test, threshold=0.5)
        yhat_t = (pt >= tl_thr).astype(int)
        tl_test_row = split_metrics(yt, pt, yhat_t, "test")
        tl_test_row["test_threshold"] = tl_thr
        print("\n========== MOBILENETV2 OFFICIAL TEST (same 624 images as custom CNN) ==========")
        print({k: round(v, 4) if isinstance(v, float) else v for k, v in tl_test_row.items()})
        print_clf_report(yt, yhat_t)
        cm = confusion_matrix(yt, yhat_t)
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES,
            ax=axes[0],
        )
        axes[0].set_title("Confusion matrix (MobileNetV2 official test)")
        axes[0].set_xlabel("Predicted")
        axes[0].set_ylabel("True")
        fpr, tpr, _ = roc_curve(yt, pt)
        axes[1].plot(fpr, tpr, label=f"AUC = {roc_auc_score(yt, pt):.3f}")
        axes[1].plot([0, 1], [0, 1], "--", color="gray")
        axes[1].set_title("ROC (MobileNetV2 official test)")
        axes[1].legend()
        plt.tight_layout()
        plt.show()

        if cnn_test_row is not None:
            both = pd.DataFrame(
                [
                    {"model": "Selected custom CNN", **cnn_test_row},
                    {"model": "MobileNetV2 (frozen)", **tl_test_row},
                ]
            )
            print("\n========== CUSTOM CNN vs MOBILENETV2 (official test, same images) ==========")
            print(both.to_string(index=False))
            both.to_csv(ckpt_dir / "official_test_cnn_vs_mobilenet.csv", index=False)
            print("Saved", ckpt_dir / "official_test_cnn_vs_mobilenet.csv")


## 7. 70/15/15 split comparison (end block)

Run this **after** 80/20 CNN, official test, and MobileNet 80/20.

- Same **baseline CNN settings** as run 1, but on a pooled 70/15/15 split.
- If `RUN_TRANSFER_LEARNING` is on, MobileNetV2 is also trained on 70/15/15.
- Own-test images are **not** the official Kaggle test. Do not mix with section 5 / 6 scores.

Set `RUN_SPLIT_COMPARISON = True` in the config cell to enable this block.

In [ ]:
# -----------------------------------------------------------------------------
# 7. All 70/15/15 comparisons (after 80/20 CNN, official test, and MobileNet 80/20)
# -----------------------------------------------------------------------------
mn70_val = None
mn70_test = None

if not RUN_SPLIT_COMPARISON:
    print("70/15/15 skipped. Set RUN_SPLIT_COMPARISON=True after the 80/20 protocol is done.")
else:
    print("\n" + "=" * 72)
    print("70/15/15 CUSTOM CNN (baseline settings; not used to pick official-test model)")
    print("=" * 72)
    row70 = train_one_run("70_15_15", "Split comparison 70/15/15", {"split_scheme": "70_15_15"})
    results_df = pd.concat([results_df, pd.DataFrame([row70])], ignore_index=True)

    print("\n========== 80/20 BASELINE vs 70/15/15 (validation; tests are DIFFERENT images) ==========")
    split_cmp = results_df[results_df["run"].astype(str).isin(["1", "70_15_15"])]
    show_now = [c for c in show_cols if c in split_cmp.columns] if "show_cols" in dir() else list(split_cmp.columns)
    print(split_cmp[show_now].to_string(index=False))
    print("Do not mix 70/15/15 own-test scores with the official Kaggle test.")

    if not RUN_TRANSFER_LEARNING:
        print("\nMobileNet 70/15/15 skipped (RUN_TRANSFER_LEARNING=False).")
    else:
        print("\n" + "=" * 72)
        print("MOBILENETV2 70/15/15 (own test is NOT official Kaggle test)")
        print("=" * 72)
        tr70, va70, te70, name70 = make_70_15_15()
        print_split_table(name70 + " (MobileNetV2)", tr70, va70, te70)
        cw70 = class_weights_from_train(tr70)
        ds_tr = make_rgb_dataset_from_paths(
            tr70["path"].tolist(), tr70["label"].tolist(), True, BATCH_SIZE, TL_IMG_SIZE
        )
        ds_va = make_rgb_dataset_from_paths(
            va70["path"].tolist(), va70["label"].tolist(), False, BATCH_SIZE, TL_IMG_SIZE
        )
        ds_te = make_rgb_dataset_from_paths(
            te70["path"].tolist(), te70["label"].tolist(), False, BATCH_SIZE, TL_IMG_SIZE
        )
        keras.backend.clear_session()
        tf.keras.utils.set_random_seed(SEED)
        tl_base = keras.applications.MobileNetV2(
            include_top=False, weights="imagenet", input_shape=(TL_IMG_SIZE, TL_IMG_SIZE, 3)
        )
        tl_base.trainable = False
        inputs = keras.Input(shape=(TL_IMG_SIZE, TL_IMG_SIZE, 3), name="xray_rgb")
        x = make_augmentation()(inputs)
        x = keras.applications.mobilenet_v2.preprocess_input(x)
        x = tl_base(x, training=False)
        x = layers.GlobalAveragePooling2D(name="gap")(x)
        x = layers.Dropout(0.3, name="dropout")(x)
        outputs = layers.Dense(1, activation="sigmoid", name="pneumonia_prob")(x)
        tl70 = keras.Model(inputs, outputs, name="mobilenetv2_70_15_15")
        tl70.compile(
            optimizer=keras.optimizers.Adam(1e-4),
            loss="binary_crossentropy",
            metrics=[
                keras.metrics.BinaryAccuracy(name="accuracy"),
                keras.metrics.Precision(name="precision"),
                keras.metrics.Recall(name="recall"),
                keras.metrics.AUC(name="auc"),
            ],
        )
        tl70_ckpt = ckpt_dir / "mobilenetv2_70_15_15_best.keras"
        t0 = time.perf_counter()
        hist70 = tl70.fit(
            ds_tr,
            validation_data=ds_va,
            epochs=TL_EPOCHS,
            class_weight=cw70,
            callbacks=[
                keras.callbacks.EarlyStopping(
                    monitor="val_loss", patience=3, restore_best_weights=True, verbose=1
                ),
                keras.callbacks.ReduceLROnPlateau(
                    monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1
                ),
                keras.callbacks.ModelCheckpoint(
                    filepath=str(tl70_ckpt), monitor="val_loss", save_best_only=True, verbose=0
                ),
            ],
            verbose=1,
        )
        mn70_seconds = float(time.perf_counter() - t0)
        pd.DataFrame(hist70.history).to_csv(ckpt_dir / "mobilenetv2_70_15_15_history.csv", index=False)

        yv, pv, _ = collect_labels(tl70, ds_va, threshold=0.5)
        thr70, thr70_info = select_threshold(yv, pv)
        yhat = (pv >= thr70).astype(int)
        mn70_val = split_metrics(yv, pv, yhat, "val")
        mn70_val["threshold"] = thr70
        mn70_val["training_seconds"] = round(mn70_seconds, 1)
        print("\nMobileNetV2 70/15/15 validation:")
        print(f"threshold={thr70:.4f}  {thr70_info}")
        print({k: round(v, 4) if isinstance(v, float) else v for k, v in mn70_val.items()})
        print_clf_report(yv, yhat)

        yt, pt, _ = collect_labels(tl70, ds_te, threshold=0.5)
        yhat_t = (pt >= thr70).astype(int)
        mn70_test = split_metrics(yt, pt, yhat_t, "own_test")
        mn70_test["own_test_threshold"] = thr70
        print("\nMobileNetV2 70/15/15 own-test (NOT official Kaggle test):")
        print({k: round(v, 4) if isinstance(v, float) else v for k, v in mn70_test.items()})
        print_clf_report(yt, yhat_t)
        pd.DataFrame([mn70_val]).to_csv(ckpt_dir / "mobilenetv2_70_15_15_val.csv", index=False)


## 8. Final summary

One table after every protocol has finished.

- **80/20 test** = official Kaggle test (624), filled only when `EVALUATE_TEST_AFTER_ALL_RUNS=True`.
- **70/15/15** rows appear only if section 7 ran.
- Compare CNN vs MobileNet on **80/20 official test** only. 70/15/15 uses different test images.


In [ ]:
# -----------------------------------------------------------------------------
# 8. Final summary: 80/20 first, then optional 70/15/15 rows from section 7
# -----------------------------------------------------------------------------
def _metric(d, *keys, default=float("nan")):
    if not isinstance(d, dict):
        return default
    for k in keys:
        if k in d and pd.notna(d[k]):
            return d[k]
    return default


def _summary_row(model, split, test_name, val_d, test_d, test_prefix="test"):
    return {
        "model": model,
        "split": split,
        "test_set": test_name,
        "threshold": _metric(val_d, "threshold", "test_threshold"),
        "training_seconds": _metric(val_d, "training_seconds"),
        "val_accuracy": _metric(val_d, "val_accuracy"),
        "val_recall": _metric(val_d, "val_recall"),
        "val_specificity": _metric(val_d, "val_specificity"),
        "val_f1": _metric(val_d, "val_f1"),
        "val_auc": _metric(val_d, "val_auc"),
        "test_accuracy": _metric(test_d, f"{test_prefix}_accuracy"),
        "test_recall": _metric(test_d, f"{test_prefix}_recall"),
        "test_specificity": _metric(test_d, f"{test_prefix}_specificity"),
        "test_f1": _metric(test_d, f"{test_prefix}_f1"),
        "test_auc": _metric(test_d, f"{test_prefix}_auc"),
    }


cnn80_val = {}
cnn80_label = "Custom CNN (80/20)"
cnn70_val, cnn70_test = {}, {}
if "results_df" in dir() and len(results_df):
    r80 = results_df[results_df["run"].astype(str) == "1"]
    r70 = results_df[results_df["run"].astype(str) == "70_15_15"]
    if "winner" in dir() and winner is not None:
        w = results_df[results_df["run"].astype(str) == str(winner["run"])]
        if len(w):
            cnn80_val = w.iloc[0].to_dict()
            cnn80_label = f"Custom CNN (80/20, run {winner['run']})"
        elif len(r80):
            cnn80_val = r80.iloc[0].to_dict()
            cnn80_label = "Custom CNN (80/20, run 1)"
    elif len(r80):
        cnn80_val = r80.iloc[0].to_dict()
        cnn80_label = "Custom CNN (80/20, run 1)"
    if len(r70):
        cnn70_val = r70.iloc[0].to_dict()
        cnn70_test = cnn70_val

summary_rows = [
    _summary_row(
        cnn80_label,
        "80/20",
        "official Kaggle 624",
        cnn80_val,
        cnn_test_row if cnn_test_row else {},
        test_prefix="test",
    )
]
if cnn70_val:
    summary_rows.append(
        _summary_row(
            "Custom CNN (baseline settings)",
            "70/15/15",
            "own 15% (not official)",
            cnn70_val,
            cnn70_test,
            test_prefix="own_test",
        )
    )
if tl_val_row:
    summary_rows.append(
        _summary_row(
            "MobileNetV2 (frozen)",
            "80/20",
            "official Kaggle 624",
            tl_val_row or {},
            tl_test_row or {},
            test_prefix="test",
        )
    )
if "mn70_val" in dir() and mn70_val:
    summary_rows.append(
        _summary_row(
            "MobileNetV2 (frozen)",
            "70/15/15",
            "own 15% (not official)",
            mn70_val or {},
            mn70_test or {},
            test_prefix="own_test",
        )
    )
summary_df = pd.DataFrame(summary_rows)
print("\n========== FINAL SUMMARY ==========")
print("80/20 custom-CNN row uses the selected winner's validation metrics, not always Run 1.")
print(summary_df.round(4).to_string(index=False))
summary_df.to_csv(ckpt_dir / "final_summary.csv", index=False)
print("Saved", ckpt_dir / "final_summary.csv")
print(
    "Official test stays empty until EVALUATE_TEST_AFTER_ALL_RUNS=True. "
    "70/15/15, if trained, is a split check with different test images."
)
try:
    from IPython.display import display

    display(summary_df.round(4))
except Exception:
    pass
